In [1]:
from vllm import LLM, SamplingParams
from transformers import AutoProcessor

import pandas as pd
import numpy as np

from tqdm import tqdm
from PIL import Image
from collections import defaultdict
from bs4 import BeautifulSoup

import torch
import glob
import os
import re

## Data Prep

In [2]:
prompt = """Extract all text from the image.

Instructions:
- Only return the clean Markdown.
- Do not include any explanation or extra text.
- You must include all information on the page.

Formatting Rules:
- Tables: Render tables using <table>...</table> in clean HTML format.
- Equations: Render equations using LaTeX syntax with inline ($...$) and block ($$...$$).
- Images/Charts/Diagrams: Wrap any clearly defined visual areas (e.g. charts, diagrams, pictures) in:

<figure>
Describe the image's main elements (people, objects, text), note any contextual clues (place, event, culture), mention visible text and its meaning, provide deeper analysis when relevant (especially for financial charts, graphs, or documents), comment on style or architecture if relevant, then give a concise overall summary. Describe in Thai.
</figure>

- Page Numbers: Wrap page numbers in <page_number>...</page_number> (e.g., <page_number>14</page_number>).
- Checkboxes: Use ☐ for unchecked and ☑ for checked boxes."""


In [3]:
def resize_if_needed(path, max_size):
    img = Image.open(path)
    width, height = img.size
    # Only resize if one dimension exceeds max_size
    if width > 300 or height > 300:
        if width >= height:
            scale = max_size / float(width)
            new_size = (max_size, int(height * scale))
        else:
            scale = max_size / float(height)
            new_size = (int(width * scale), max_size)

        img = img.resize(new_size, Image.Resampling.LANCZOS)
        # print(f"{width, height}==> {img.size}")
        return img
    else:
        return img 

image_files = np.sort(glob.glob("./final_data/images/*.png"))
img_dict = {}
for path in tqdm(image_files):
    fname = os.path.basename(path)[:-4]
    img = resize_if_needed(path, max_size=1800)
    img_dict[fname] = img

100%|██████████| 846/846 [03:15<00:00,  4.34it/s]


## Inference

In [4]:
# 1. Load model with multimodal support
model_id = "./typhoon-ocr-2b"
llm = LLM(
    model                  = model_id,
    gpu_memory_utilization = 0.8,
    max_model_len          = 32000,
    limit_mm_per_prompt    = {"image": 4},     # ← very important: how many images per prompt
    enforce_eager          = True,
)

INFO 03-22 04:46:07 [utils.py:238] non-default args: {'max_model_len': 32000, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 4}, 'model': './typhoon-ocr-2b'}
INFO 03-22 04:46:07 [model.py:531] Resolved architecture: Qwen3VLForConditionalGeneration
INFO 03-22 04:46:07 [model.py:1554] Using max model len 32000
INFO 03-22 04:46:07 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-22 04:46:07 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 03-22 04:46:07 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 03-22 04:46:07 [vllm.py:792] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 03-22 04:46:09 [vllm.py:957] Cudagraph is disabled under eager mode
(EngineCore_DP0 pid=2965820) INFO 03-2

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore_DP0 pid=2965820) INFO 03-22 04:46:18 [default_loader.py:293] Loading weights took 3.59 seconds
(EngineCore_DP0 pid=2965820) INFO 03-22 04:46:18 [gpu_model_runner.py:4338] Model loading took 4.31 GiB memory and 4.321883 seconds
(EngineCore_DP0 pid=2965820) INFO 03-22 04:46:19 [gpu_model_runner.py:5254] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(EngineCore_DP0 pid=2965820) INFO 03-22 04:46:26 [gpu_worker.py:424] Available KV cache memory: 25.35 GiB
(EngineCore_DP0 pid=2965820) INFO 03-22 04:46:26 [kv_cache_utils.py:1314] GPU KV cache size: 237,328 tokens
(EngineCore_DP0 pid=2965820) INFO 03-22 04:46:26 [kv_cache_utils.py:1319] Maximum concurrency for 32,000 tokens per request: 7.42x
(EngineCore_DP0 pid=2965820) INFO 03-22 04:46:27 [core.py:282] init engine (profile, create kv cache, warmup model) took 8.16 seconds
(EngineCore_DP0 pid=2965820) WARNING 03-22 04:46:27 [vllm.py:781] Enforce eager 

In [5]:
processor = AutoProcessor.from_pretrained("./typhoon-ocr-2b")

conversations = []
for fname, img in tqdm(img_dict.items()):
    messages = [{
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt}
                ]}
               ]
    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    conversations.append({
        "prompt": prompt_text,
        "multi_modal_data": {"image": img}
    })

print(conversations[0])

100%|██████████| 846/846 [00:00<00:00, 15188.82it/s]

{'prompt': "<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>Extract all text from the image.\n\nInstructions:\n- Only return the clean Markdown.\n- Do not include any explanation or extra text.\n- You must include all information on the page.\n\nFormatting Rules:\n- Tables: Render tables using <table>...</table> in clean HTML format.\n- Equations: Render equations using LaTeX syntax with inline ($...$) and block ($$...$$).\n- Images/Charts/Diagrams: Wrap any clearly defined visual areas (e.g. charts, diagrams, pictures) in:\n\n<figure>\nDescribe the image's main elements (people, objects, text), note any contextual clues (place, event, culture), mention visible text and its meaning, provide deeper analysis when relevant (especially for financial charts, graphs, or documents), comment on style or architecture if relevant, then give a concise overall summary. Describe in Thai.\n</figure>\n\n- Page Numbers: Wrap page numbers in <page_number>...</page_number> (e.g., <page_numb

In [6]:
# 3. Generate (batched version accepts list of conversations)
sampling_params = SamplingParams(
    temperature=0.0, top_p=1, top_k=-1,
    max_tokens=32000,
    skip_special_tokens=True,
)

outputs = llm.generate(conversations, sampling_params=sampling_params)

ocr_results = []
for o in outputs:
    generated_text = o.outputs[0].text
    ocr_results.append(generated_text)

Rendering prompts:   0%|          | 0/846 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/846 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

## Extract result

In [7]:
def extract_table_from_html(raw_html):
    # 1. Parse the HTML content
    soup = BeautifulSoup(raw_html, 'html.parser')
    
    # 2. Find the table tag
    table = soup.find('table')
    if not table:
        return None

    data = []
    rows = table.find_all('tr')
    
    for row in rows:
        # 3. Extract cells (td) from each row
        cols = row.find_all('td')
        # Clean whitespace and strip newline characters
        cols = [ele.text.strip() for ele in cols]
        data.append(cols)
    
    return data

def normalize_columns(columns):
    # Mapping: { "Global Name": [Keywords to look for] }
    mapping = {
        'พรรคการเมือง': ['พรรคการเมือง', 'พรรค การเมือง', 'พรรค/การเมือง', 'สังกัด'],
        'ชื่อผู้สมัคร': ['ชื่อตัว', 'ชื่อ - สกุล', 'ชื่อสกุล', 'ผู้สมัครรับเลือกตั้ง'],
        'คะแนน': ['ได้คะแนน', 'คะแนน', 'หนึ่งพัน'], # Added common OCR text found in scores
        'หมายเลข': ['หมายเลข ประจำตัว', 'หมายเลขประจำตัว', 'หมายเลข'],
        'ลำดับ': ['ลำดับที่']
    }
    
    new_cols = []
    for col in columns:
        col_clean = str(col).strip()
        matched = False
        for standard_name, keywords in mapping.items():
            if any(key in col_clean for key in keywords):
                new_cols.append(standard_name)
                matched = True
                break
        if not matched:
            new_cols.append(col_clean)
    return new_cols

In [10]:
final_df = pd.DataFrame()
fnames = list(img_dict.keys())

for fname, text in tqdm(zip(fnames, ocr_results), total=len(fnames)):
    table_data = extract_table_from_html(text)
    
    # 1. Handle cases where a table was actually found
    if table_data and len(table_data) > 0:
        header = table_data[0]
        raw_rows = table_data[1:]
        
        # FIX: Align row length with header length to prevent ValueError
        num_cols = len(header)
        cleaned_rows = []
        for r in raw_rows:
            cleaned_rows.append(r[:num_cols] + [None] * (num_cols - len(r)))

        temp_df = pd.DataFrame(cleaned_rows, columns=header)
        
        # 2. Normalize headers to global names
        temp_df.columns = normalize_columns(temp_df.columns)
        
        # Merge duplicate columns (e.g., if two messy headers both mapped to 'พรรคการเมือง')
        temp_df = temp_df.loc[:, ~temp_df.columns.duplicated()].copy()
        
        temp_df['doc_id'] = fname
        
    # 3. Handle cases where NO table was found (Your Else Condition)
    else:
        # Create a single row with just the doc_id; other columns will be NaN later
        temp_df = pd.DataFrame({'doc_id': [fname]})
    
    # 4. Append to final result
    final_df = pd.concat([final_df, temp_df], ignore_index=True, sort=False)

# Final cleanup: Move doc_id to the first column
if not final_df.empty:
    cols = ['doc_id'] + [c for c in final_df.columns if c != 'doc_id']
    final_df = final_df[cols]

print(f"Final Data Shape: {final_df.shape}")

100%|██████████| 846/846 [00:03<00:00, 268.85it/s]

Final Data Shape: (12452, 47)


In [19]:
final_df = final_df[~final_df.apply(lambda row: row.astype(str).str.contains('รวมคะแนน').any(), axis=1)]
final_df.shape

(12167, 47)

In [20]:
final_df[['doc_id', 'พรรคการเมือง', 'คะแนน']].to_csv("ocr_extract.csv", index=False, encoding='utf-8-sig')